# OCTOVOX — run the whole project on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnistonTechnologiesLLP/New_OCTOVOX/blob/master/OCTOVOX_Colab.ipynb)

> The badge opens this notebook straight from GitHub. The repo is private, so it only works once you're signed in to Colab with a GitHub account that has access to `AnistonTechnologiesLLP/New_OCTOVOX` (File → Open notebook → GitHub tab also works).

8-channel speech front-end for the **sensiBel SB-POLARIS** optical MEMS array. The 11-stage **production voice pipeline** (`prod_pipeline.run_production`) turns 8 raw mic channels into one clean mono voice.

### Why this notebook builds a Python 3.11 venv
Colab's kernel is **Python 3.12**, but two of the project's validated deps have **no 3.12 wheels**: `torch==2.1.2` (min 3.12 wheel is 2.2.0) and `deepfilternet==0.5.6` (tries to compile a Rust extension from source → fails). So Section 2 uses **`uv`** to build a throwaway **Python 3.11** venv — the same combo as the local `.venv311` — and the project runs *inside* it. The notebook kernel only ever reads the output WAVs to play them, so the kernel's own Python version doesn't matter.

Two ways to run:
* **Option A — Web UI**: launch the Flask app (in the 3.11 venv) and open the console via Colab's port proxy.
* **Option B — Programmatic**: drive the pipeline through a tiny runner script and A/B raw vs clean audio inline.

A **CPU runtime is enough** (the pipeline is sub-real-time and CPU-pinned by design). For optional GPU DSP see Section 3b.

---

## 1. Get the project onto Colab

The repo is private (`AnistonTechnologiesLLP/New_OCTOVOX`) and the 8-ch sample WAVs are committed in `data/input/`, so cloning brings everything you need.

### 1a. Clone from GitHub with a token (recommended)
Create a fine-grained Personal Access Token with read access to the repo (GitHub → Settings → Developer settings → PATs), then paste it below. (Alternatives — zip upload / Drive mount — are in the cell after.)

In [ ]:
# === 1a. Clone via token ===
from getpass import getpass
import os

GITHUB_TOKEN = getpass('GitHub token (input hidden): ').strip()
REPO = 'AnistonTechnologiesLLP/New_OCTOVOX'

if os.path.isdir('New_OCTOVOX'):
    print('New_OCTOVOX already present — skipping clone.')
else:
    !git clone --depth 1 https://{GITHUB_TOKEN}@github.com/{REPO}.git

%cd New_OCTOVOX
!ls -la

### 1b. Alternatives (no token)

**Upload a zip** (zip `New_OCTOVOX` locally, excluding `.venv311` and `data/output`):
```python
from google.colab import files
up = files.upload()                       # pick New_OCTOVOX.zip
import zipfile, io
with zipfile.ZipFile(io.BytesIO(up[next(iter(up))])) as z:
    z.extractall('.')
%cd New_OCTOVOX
```

**From Google Drive:**
```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/New_OCTOVOX      # adjust to your path
```

## 2. Build the Python 3.11 environment

`uv` fetches a standalone Python 3.11 and installs the validated stack into a local `.venv` (**numpy<2**, CPU **torch 2.1.2**, **DeepFilterNet3**, **nara-wpe**). PortAudio is added so `sounddevice` imports. First run takes a few minutes; the `.venv` persists for the rest of the session.

In [ ]:
# --- system lib for sounddevice ---
!apt-get -qq install -y libportaudio2 > /dev/null && echo 'libportaudio2 ok'

# --- uv + a standalone Python 3.11, then a local .venv ---
!pip install -q uv
!uv python install 3.11
!uv venv --python 3.11 .venv
PY = '.venv/bin/python'

# --- CPU torch first (so DeepFilterNet reuses it instead of pulling another) ---
!uv pip install -p {PY} torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cpu

# --- core DSP/web stack + neural extras (numpy MUST stay <2) ---
!uv pip install -p {PY} 'numpy>=1.20,<2' 'scipy>=1.7' 'matplotlib>=3.4' 'flask>=2.0' \
    sounddevice 'soundfile>=0.12' deepfilternet==0.5.6 nara-wpe

# --- re-pin numpy in case a dep bumped it ---
!uv pip install -p {PY} 'numpy>=1.20,<2'
print('\nvenv ready at .venv (Python 3.11)')

## 3. Sanity check (inside the venv)

Confirms the optional neural deps loaded under the 3.11 interpreter. The env vars CPU-pin the neural stack and force UTF-8 so the banner can't crash.

In [ ]:
import subprocess, os
check = r'''
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
import numpy, scipy
from octovox_app.services import pipeline as ov
print("python      :", __import__("sys").version.split()[0])
print("numpy/scipy :", numpy.__version__, scipy.__version__)
print("DeepFilterNet:", ov.HAS_DFN, "| nara-wpe:", ov.HAS_WPE, "| Silero VAD:", ov.HAS_VAD)
assert ov.HAS_DFN and ov.HAS_WPE and ov.HAS_VAD, "an optional dep failed to load — see Section 2"
print("\nAll components ready.")
'''
# MPLBACKEND=Agg: Colab sets MPLBACKEND to an inline backend that only exists in
# the kernel, not the venv — the project uses Agg anyway.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
proc = subprocess.run(['.venv/bin/python', '-c', check], env=env, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

## 3b. (optional) GPU acceleration — CuPy DSP

**Skip on a CPU runtime.** On a GPU runtime (Runtime → Change runtime type → **T4 GPU**), CuPy moves the heaviest linear algebra — the PAST RTF tracker, its time-varying MVDR, and the masked covariance build — onto the GPU (~12–100× vs the CPU loops). The neural stack (DeepFilterNet3 + Silero) **stays on CPU** by design: `pipeline.py` imports CuPy first (grabbing the GPU for DSP) and only then hides CUDA from torch, which is how the project gets GPU DSP without the DFN CUDA/CPU crash.

Run this to add CuPy to the venv, then launch the server / runner **without** setting `CUDA_VISIBLE_DEVICES=""` (the launch cells below read `OCTOVOX_GPU` to decide). The win shows on `beam='tracked'`; the default `beam='auto'` usually picks the batch beamformer, which stays on CPU on purpose (it needs SciPy's generalized `eigh`, which CuPy lacks).

In [ ]:
import os, subprocess

# Colab GPU runtimes are CUDA 12.x. [ctk] ships the toolkit headers CuPy JITs against.
!uv pip install -p .venv/bin/python "cupy-cuda12x[ctk]"

os.environ['OCTOVOX_GPU'] = '1'   # the launch cells below will NOT pin CUDA off when this is '1'

# Verify the GPU is visible to CuPy (and that torch stays CPU) inside the venv.
verify = r'''
import cupy as cp                       # imported BEFORE the pipeline, like pipeline.py does
n = cp.cuda.runtime.getDeviceCount()
print("CuPy device count:", n)
if n:
    print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
from octovox_app.services import pipeline as ov
print("HAS_CUPY:", ov.HAS_CUPY, "| GPU_DSP:", ov.GPU_DSP, "| torch CUDA:", ov.HAS_CUDA)
'''
env = dict(os.environ, OCTOVOX_GPU='1', PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
env.pop('CUDA_VISIBLE_DEVICES', None)   # must NOT be '' or CuPy can't see the GPU
proc = subprocess.run(['.venv/bin/python', '-c', verify], env=env, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    print('\nGPU not visible to CuPy. Check Runtime → Change runtime type → T4 GPU, then '
          're-run from Section 2. The pipeline still works on CPU regardless.')

---
## Option A — Run the Web UI inside Colab

Run the **two cells below in order**: the first writes `octovox_colab_serve.py` (the project's server plus a numpy-safe JSON provider — without it `/api/clean` raises *"Object of type float32 is not JSON serializable"* in the browser), the second launches it in the 3.11 venv and opens the console via Colab's port proxy (no ngrok / external account).

In the UI: pick a recording in **02 · Files**, set the knobs, analyse, then A/B raw vs clean in **03 · Console**. For GPU DSP, run **Section 3b** before these cells and set **Beam → tracked**.

In [ ]:
%%writefile octovox_colab_serve.py
# Launcher used instead of run.py on Colab: identical server, plus a numpy-aware
# JSON provider so /api/clean can serialize the np.float32/np.int values some
# pipeline stages emit (Flask's default JSON encoder cannot, which surfaces as
# "Object of type float32 is not JSON serializable" in the browser).
import os
os.environ['MPLBACKEND'] = 'Agg'                 # Colab's inline backend doesn't exist in the venv
# CUDA pinning is set by the launching environment (CPU vs GPU mode).

import numpy as np
from flask.json.provider import DefaultJSONProvider
# create_app imports the pipeline (CuPy-before-torch); do NOT import torch above this.
from octovox_app import create_app, config
from octovox_app.services import pipeline as ov


class NumpyJSONProvider(DefaultJSONProvider):
    @staticmethod
    def default(o):
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return DefaultJSONProvider.default(o)


app = create_app()
app.json = NumpyJSONProvider(app)
try:
    ov.warm_up_models()
except Exception:
    pass
print('OCTOVOX serving on %s:%s (numpy-safe JSON)' % (config.HOST, config.PORT), flush=True)
app.run(host=config.HOST, port=config.PORT, threaded=True, debug=False, use_reloader=False)

In [ ]:
import subprocess, os, time, pathlib, urllib.request

gpu_mode = os.environ.get('OCTOVOX_GPU', '0') == '1'
env = dict(os.environ, PYTHONIOENCODING='utf-8', OCTOVOX_HOST='0.0.0.0',
           OCTOVOX_PORT='5050', MPLBACKEND='Agg')
if gpu_mode:
    env.pop('CUDA_VISIBLE_DEVICES', None)   # GPU mode: let CuPy see the device
else:
    env['CUDA_VISIBLE_DEVICES'] = ''        # CPU mode: pin neural stack to CPU

# Report the DSP backend the server will use. NOTE: the web UI / /api/env only
# shows torch's CUDA state, which is ALWAYS False here — the neural stack
# (DeepFilterNet + Silero) is CPU-pinned by design. GPU acceleration is CuPy on
# the DSP, confirmed below. The GPU win shows on beam='tracked'.
if gpu_mode:
    chk = subprocess.run(['.venv/bin/python', '-c',
        'import cupy as cp; print(cp.cuda.runtime.getDeviceCount(),'
        ' cp.cuda.runtime.getDeviceProperties(0)[\"name\"].decode())'],
        env=env, text=True, capture_output=True)
    print('DSP backend: GPU (CuPy) —', chk.stdout.strip() or chk.stderr.strip())
else:
    print('DSP backend: CPU  (run Section 3b first for GPU DSP)')

# Launch via octovox_colab_serve.py (run.py + a numpy-safe JSON provider). Make
# sure the launcher cell above has been run so the file exists.
logf = open('octovox_server.log', 'w')
server = subprocess.Popen(['.venv/bin/python', 'octovox_colab_serve.py'], env=env,
                          stdout=logf, stderr=subprocess.STDOUT)

print('starting OCTOVOX (loading neural models, ~20-40s the first time)…')
ready = False
for _ in range(90):
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen('http://127.0.0.1:5050/', timeout=2); ready = True; break
    except Exception:
        time.sleep(1)

print('\n--- server log ---')
print(pathlib.Path('octovox_server.log').read_text()[-1500:])

if ready:
    from google.colab.output import serve_kernel_port_as_window, serve_kernel_port_as_iframe
    serve_kernel_port_as_window(5050)                       # opens in a new tab
    # serve_kernel_port_as_iframe(5050, height='800')       # or embed inline instead
else:
    print('\nServer did not come up — check the log above.')

Stop the server when done: `server.terminate()`.

---
## Option B — Run the pipeline programmatically

A tiny runner (`octovox_colab_run.py`) executes the chain inside the 3.11 venv and prints the result paths as JSON. The notebook kernel then plays the WAVs inline — reading a WAV needs no neural deps, so the kernel's Python 3.12 is fine.

First, write the runner:

In [ ]:
%%writefile octovox_colab_run.py
import os, sys, json

# GPU mode (OCTOVOX_GPU=1): do NOT pre-hide CUDA — pipeline.py imports CuPy first
# (grabbing the GPU for DSP) and only then pins torch to CPU itself. CPU mode:
# hide CUDA so the neural stack stays on CPU.
if os.environ.get('OCTOVOX_GPU', '0') != '1':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
os.environ['MPLBACKEND'] = 'Agg'                     # force-override Colab's inline backend

import numpy as np
from octovox_app import config
from octovox_app.services import prod_pipeline as prod


# JSON fallback for the numpy scalars/arrays the pipeline returns.
def _ser(o):
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError('not serializable: ' + str(type(o)))


config.ensure_dirs()
fname = sys.argv[1]
kw = json.loads(sys.argv[2]) if len(sys.argv) > 2 else {}
res = prod.run_production(config.INPUT_DIR / fname, config.OUTPUT_DIR, **kw)
res['_stem_dir'] = str(config.OUTPUT_DIR / res['stem'])
print('OCTOVOX_RESULT_JSON:' + json.dumps(res, default=_ser))

In [ ]:
# List the bundled recordings (kernel just globs the folder — no deps needed)
from pathlib import Path
inputs = sorted(Path('data/input').glob('*.wav'))
for i, p in enumerate(inputs):
    print(f'{i:2d}  {p.name}')

In [ ]:
import subprocess, os, json
import IPython.display as ipd

# Pick a recording and the knobs (mirror /api/clean):
#   nr='dfn'|'fast'|'none'   beam='auto'|'batch'|'tracked'
#   dereverb='none'|'spectral'|'wpe'   movement='srp'|'rtf'
#   residual=0..1   mvdr_blend=0..1   eq=True/False
filename = inputs[0].name
knobs = {'nr': 'dfn', 'beam': 'auto', 'dereverb': 'none', 'residual': 0.6, 'eq': True}

env = dict(os.environ, PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
if os.environ.get('OCTOVOX_GPU', '0') == '1':
    env.pop('CUDA_VISIBLE_DEVICES', None)   # GPU mode: let CuPy see the device
else:
    env['CUDA_VISIBLE_DEVICES'] = ''        # CPU mode: pin neural stack to CPU

proc = subprocess.run(['.venv/bin/python', 'octovox_colab_run.py', filename, json.dumps(knobs)],
                      env=env, text=True, capture_output=True)
if proc.returncode != 0:
    print(proc.stdout); print(proc.stderr); raise SystemExit('pipeline failed — see traceback above')

res = json.loads([l for l in proc.stdout.splitlines() if l.startswith('OCTOVOX_RESULT_JSON:')][-1]
                 [len('OCTOVOX_RESULT_JSON:'):])
sd = Path(res['_stem_dir'])
print(f"cleaned '{filename}' in {res['elapsed_s']:.2f}s  ({res['sr']} Hz, {res['n_channels']} ch)\n")
print('--- per-stage timing (ms) ---')
for stage, ms in res['timings'].items():
    print(f'  {stage:24s} {ms:8.1f}')

print('\nRAW (8-ch downmix):'); ipd.display(ipd.Audio(str(sd / res['input_name'])))
print('CLEAN (production output):'); ipd.display(ipd.Audio(str(sd / res['clean_name'])))

In [ ]:
# Show the standalone A/B report (waveforms, spectrograms, KPIs, diagnostics)
from IPython.display import HTML
rep = sd / res['report_name'] if res.get('report_name') else None
HTML(rep.read_text(encoding='utf-8')) if rep else print('no report generated')

### Batch: clean every bundled recording (optional)

In [ ]:
import subprocess, os, json
env = dict(os.environ, PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
if os.environ.get('OCTOVOX_GPU', '0') == '1':
    env.pop('CUDA_VISIBLE_DEVICES', None)
else:
    env['CUDA_VISIBLE_DEVICES'] = ''
knobs = json.dumps({'nr': 'dfn', 'beam': 'auto'})
for p in inputs:
    proc = subprocess.run(['.venv/bin/python', 'octovox_colab_run.py', p.name, knobs],
                          env=env, text=True, capture_output=True)
    if proc.returncode != 0:
        print('FAILED', p.name); print(proc.stderr[-500:]); continue
    r = json.loads([l for l in proc.stdout.splitlines() if l.startswith('OCTOVOX_RESULT_JSON:')][-1]
                   [len('OCTOVOX_RESULT_JSON:'):])
    print(f"{r['elapsed_s']:6.2f}s  {p.name}  ->  {r['stem']}/{r['clean_name']}")

### Process your own recording
Upload an 8-channel (48 kHz) WAV; it's saved into `data/input/`. Re-run the **Option B** run cell with `filename` set to it (fewer channels also work — the pipeline downmixes).

In [ ]:
from google.colab import files
import shutil
from pathlib import Path
up = files.upload()
for name in up:
    dest = Path('data/input') / name
    shutil.move(name, dest)
    print('saved ->', dest)
    filename = name   # point the Option-B run cell at this file

## 4. Download / keep the results
Outputs land in `data/output/<stem>/` (`clean_prod.wav`, `input_mono.wav`, `report.html`, `visualization.png`).

In [ ]:
# Zip + download the last processed recording's outputs
import shutil
from google.colab import files
zip_base = f"/content/{res['stem']}_outputs"
shutil.make_archive(zip_base, 'zip', str(sd))
files.download(zip_base + '.zip')

# Or copy the whole output tree to Drive:
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copytree('data/output', '/content/drive/MyDrive/octovox_output', dirs_exist_ok=True)